In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd

from src import config
from src.console import print_header, print_kv, print_status
from src.error_analysis import load_run_predictions
from src.model_analysis import (evaluate_branch_ablation, load_run_embeddings, load_run_gate_weights,
                                plot_branch_ablation, plot_complementarity_heatmap, plot_embedding_map,
                                plot_gate_by_severity, plot_gate_distribution, plot_permutation_importance,
                                plot_shap_group_importance, plot_shap_per_class, plot_shap_summary,
                                compute_permutation_importance, aggregate_permutation_importance_by_group,
                                compute_shap_values, summarize_gate_values)
from src.praat import FEATURE_GROUPS, SEGMENTAL_FEATURE_COLUMNS, SUPRASEGMENTAL_FEATURE_COLUMNS, load_praat_table
from src.results import select_analysis_run
from src.splits import iter_severity_loso_folds
from src.training.checkpoint import load_checkpoint
from src.training.data import build_loaders, load_manifest
from src.training.models import SEVERITY_MODEL_NAME, build_model
from src.training.utils import resolve_device

config.ensure_directories()
TASK = "severity"

df_m6 = load_manifest()
RUN_NAME = select_analysis_run(task=TASK, metric="f1", preferred="severity_gated_fusion_three_branch")
print_header("Three-Branch Model Analysis")
print_kv("Analysis run", RUN_NAME or "none eligible yet -- run notebooks/03_training.ipynb first")

In [ ]:
device = resolve_device(None)
model = None

if RUN_NAME is not None:
    ckpt_dir = config.CHECKPOINT_DIR / RUN_NAME
    fold_dirs = sorted(p.parent.name for p in ckpt_dir.glob("*/best.pt")) if ckpt_dir.exists() else []
    if fold_dirs:
        example_fold = fold_dirs[0]
        model = build_model(SEVERITY_MODEL_NAME, config.NUM_CLASSES[TASK], num_speakers=1).to(device)
        load_checkpoint(ckpt_dir / example_fold / "best.pt", model, map_location=str(device))
        model.eval()
        print_status(f"Loaded checkpoint: {RUN_NAME}/{example_fold}/best.pt", ok=True)
    else:
        print_status("No checkpoint found for this run yet.", ok=False)
else:
    print_status("No eligible run yet -- nothing to load.", ok=False)

In [ ]:
preds = None
if RUN_NAME is not None:
    try:
        preds = load_run_predictions(RUN_NAME)
    except FileNotFoundError as e:
        print_status(str(e), ok=False)
preds.head() if preds is not None else None

In [ ]:
branch_embeddings = {}
if RUN_NAME is not None:
    for branch in ("learned", "segmental", "supra", "fused"):
        try:
            branch_embeddings[branch] = load_run_embeddings(RUN_NAME, embedding_type=branch)
        except (FileNotFoundError, ValueError) as e:
            print_status(f"{branch}: {e}", ok=False)
print_kv("Representation sets loaded", list(branch_embeddings))

In [ ]:
for branch, df in branch_embeddings.items():
    matrix = np.vstack(df["embedding"].to_numpy())
    print_kv(f"{branch} embeddings", f"shape={matrix.shape}  mean={matrix.mean():.4f}  std={matrix.std():.4f}")

In [ ]:
expected_dims = {"learned": config.LEARNED_EMBED_DIM, "segmental": config.SEGMENTAL_EMBED_DIM,
                 "supra": config.SUPRA_EMBED_DIM, "fused": config.FUSED_EMBED_DIM}
for branch, df in branch_embeddings.items():
    actual_dim = np.vstack(df["embedding"].to_numpy()).shape[1]
    assert actual_dim == expected_dims[branch], f"{branch}: expected {expected_dims[branch]}, got {actual_dim}"
if branch_embeddings:
    print_status("Every loaded embedding set matches its frozen bottleneck dimension", ok=True)

In [ ]:
ablation_df = None
if model is not None:
    held_out_speaker, train_df, test_df = next(iter(iter_severity_loso_folds(df_m6)))
    _, _, test_loader = build_loaders(train_df, train_df.head(0), test_df, batch_size=8,
                                      num_workers=0, pin_memory=False, model_name=SEVERITY_MODEL_NAME)
    print_kv("Ablation evaluated on held-out speaker", held_out_speaker)
    ablation_df = evaluate_branch_ablation(model, test_loader, device, task=TASK)
    plot_branch_ablation(ablation_df, metric="macro_f1", show=True)
ablation_df

In [ ]:
gate_df, gate_summary = None, None
if RUN_NAME is not None:
    try:
        gate_df = load_run_gate_weights(RUN_NAME)
        gate_summary = summarize_gate_values(gate_df)
        plot_gate_distribution(gate_df, show=True)
        if preds is not None:
            merged_gate = gate_df.merge(preds[["filename", "y_true_label"]], on="filename", how="inner")
            plot_gate_by_severity(merged_gate, merged_gate["y_true_label"], show=True)
    except FileNotFoundError as e:
        print_status(str(e), ok=False)
gate_summary

In [ ]:
if RUN_NAME is not None and preds is not None and "fused" in branch_embeddings:
    plot_embedding_map(RUN_NAME, preds, task=TASK, method="pca", embedding_type="fused",
                       color_by="severity", show=True)

In [ ]:
try:
    import umap  # noqa: F401
    if RUN_NAME is not None and preds is not None and "fused" in branch_embeddings:
        plot_embedding_map(RUN_NAME, preds, task=TASK, method="umap", embedding_type="fused",
                           color_by="severity", show=True)
except ImportError:
    print_status("umap-learn not installed -- skipping UMAP views.", ok=False)

In [ ]:
if RUN_NAME is not None and preds is not None:
    for branch in branch_embeddings:
        plot_embedding_map(RUN_NAME, preds, task=TASK, method="tsne", embedding_type=branch,
                           color_by="severity", show=True)

In [ ]:
if RUN_NAME is not None and preds is not None:
    for branch in branch_embeddings:
        try:
            plot_embedding_map(RUN_NAME, preds, task=TASK, method="tsne", embedding_type=branch,
                               color_by="speaker", show=True)
        except ValueError as e:
            print_status(str(e), ok=False)

In [ ]:
complementarity_table = None
if all(b in branch_embeddings for b in ("learned", "segmental", "supra")):
    z_l = np.vstack(branch_embeddings["learned"]["embedding"].to_numpy())
    z_s = np.vstack(branch_embeddings["segmental"]["embedding"].to_numpy())
    z_p = np.vstack(branch_embeddings["supra"]["embedding"].to_numpy())
    n = min(len(z_l), len(z_s), len(z_p))
    _, complementarity_table = plot_complementarity_heatmap(z_l[:n], z_s[:n], z_p[:n], show=True)
complementarity_table

In [ ]:
praat_table, shap_explanations, shap_feature_columns, class_names, surrogate = None, None, None, None, None
try:
    praat_table = load_praat_table().reset_index()
    feature_columns = list(SEGMENTAL_FEATURE_COLUMNS) + list(SUPRASEGMENTAL_FEATURE_COLUMNS)
    shap_explanations, X_sample, shap_feature_columns, class_names, surrogate = compute_shap_values(
        praat_table, task=TASK, feature_columns=feature_columns, run_name=RUN_NAME)
    # shap_explanations is a list of per-class Explanations for severity --
    # this is a RandomForest SURROGATE fit on engineered acoustic features,
    # NOT a direct explanation of the neural network's internal weights.
    plot_shap_summary(shap_explanations[-1], shap_feature_columns, show=True,
                      title=f"SHAP -- {class_names[-1]} class (surrogate model)")
except FileNotFoundError as e:
    print_status(str(e), ok=False)

In [ ]:
shap_group_df, permutation_df, permutation_group_df = None, None, None
if shap_explanations is not None:
    global_mean_abs = np.mean([np.abs(e.values).mean(axis=0) for e in shap_explanations], axis=0)
    global_table = pd.DataFrame({"feature": shap_feature_columns, "mean_abs_shap": global_mean_abs})
    global_table["group"] = global_table["feature"].map(FEATURE_GROUPS)
    shap_group_df = (global_table.groupby("group")["mean_abs_shap"].sum()
                     .sort_values(ascending=False).reset_index())
    plot_shap_group_importance(shap_group_df, show=True)

    label_map = {k: v for k, v in config.SEVERITY_LABEL_MAP.items() if v >= 0}
    labeled = praat_table[praat_table["Severity"].isin(label_map)].dropna(subset=shap_feature_columns)
    X_all = labeled[shap_feature_columns].to_numpy()
    y_all = labeled["Severity"].map(label_map).to_numpy()
    permutation_df = compute_permutation_importance(surrogate, X_all, y_all, shap_feature_columns)
    permutation_group_df = aggregate_permutation_importance_by_group(permutation_df)
    plot_permutation_importance(permutation_df, show=True)

    print_status("SHAP is a surrogate-model explanation, not a causal claim; permutation "
                "importance is an independent cross-check, and the two rankings are not "
                "guaranteed to agree.", ok=True)
permutation_df

In [ ]:
if shap_explanations is not None:
    plot_shap_per_class(shap_explanations, shap_feature_columns, class_names, show=True)

if ablation_df is not None:
    print_kv("Branch ablation, per-metric", ablation_df[["macro_f1", "ordinal_mae", "balanced_accuracy"]])

if gate_summary is not None:
    print_kv("Gate contribution summary", gate_summary)